# 05. RNDC Freight Microcase

This notebook:

1. Downloads one month of Colombian road-freight records (January 2021) from datos.gov.co.
2. Profiles the fields that matter for the analysis: waiting times, trip duration, vehicle configuration and agreed vs paid freight value.
3. Checks the unit of analysis. Each row is a consignment (*remesa*), so trips that carry several consignments appear several times.
4. Answers three operational questions with SQL (DuckDB over a Parquet file), weighting each record so that every trip counts once.

**Scope.** A one-month snapshot, so there's no trend analysis. It works as a local complement to the U.S. flights analysis in Notebooks 01–04: same kind of operational questions, applied to trucking in Colombia.

**Prerequisites.** Internet access to `datos.gov.co` and the `duckdb` package. No database server needed.

**Data source.** "Tiempos logísticos de cada viaje de vehículos de carga" (Ministerio de Transporte, RNDC), resource `tfrd-amb4`. This was the original source for this project. I dropped it as the main dataset after confirming it only contains January 2021, but one month of 836K consignment records (about 670K trips) is still enough for a cross-sectional look.

# 1. Environment setup

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import logging
import time
from pathlib import Path

import duckdb
import pandas as pd
import requests
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

BASE_URL = "https://www.datos.gov.co/resource/tfrd-amb4.json"
RAW_DIR = Path.cwd().parent / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = RAW_DIR / "rndc_trips_2021_01.parquet"

# 2. Download

The API (Socrata SODA) returns pages of up to 50,000 rows. I order by the internal row id (`:id`) so that pages don't overlap or skip rows between requests. The file is saved once; if it already exists, the cell just loads it.

Everything arrives as text, including some numeric fields, so I save the raw strings and do the typing in DuckDB.

In [2]:
PAGE_SIZE = 50_000

if RAW_PATH.exists():
    trips_raw = pd.read_parquet(RAW_PATH)
    logger.info(f"Loaded existing file: {trips_raw.shape}")
else:
    total = int(requests.get(BASE_URL, params={"$select": "count(*) as n"}, timeout=30).json()[0]["n"])
    logger.info(f"Rows reported by the API: {total:,}")

    pages = []
    for offset in tqdm(range(0, total, PAGE_SIZE), desc="Downloading"):
        params = {"$limit": PAGE_SIZE, "$offset": offset, "$order": ":id"}
        for attempt in range(3):
            try:
                response = requests.get(BASE_URL, params=params, timeout=120)
                response.raise_for_status()
                pages.append(pd.DataFrame(response.json()))
                break
            except requests.exceptions.RequestException as exc:
                logger.warning(f"Offset {offset}, attempt {attempt + 1} failed: {exc}")
                time.sleep(5)
        else:
            raise RuntimeError(f"Page at offset {offset} failed three times")
        time.sleep(0.5)

    trips_raw = pd.concat(pages, ignore_index=True).astype("string")
    trips_raw.to_parquet(RAW_PATH, index=False)
    logger.info(f"Saved {trips_raw.shape} to {RAW_PATH}")

trips_raw.head()

2026-10-27 11:34:53,180 - INFO - Loaded existing file: (836013, 26)


,a_o_mes,naturaleza,cod_producto,producto,cantidad,unid_medida,fechasalidacargue,hora_salida_cargue,fechallegadadescargue,hora_llegada_descargue,...,horas_cargue,horas_espera_descargue,horas_descargue,valor_pactado,valor_pagado,cantidad_remesas_viaje,empresa_transporte,placa,configuracion,conductor
0,202101,Carga Normal,9604,MANUFACTURAS DIVERSAS-TAMICES/CEDAZOS Y CRIBAS...,150,Kilogramos,2021-01-26T00:00:00.000,1899-12-31T19:01:00.000,2021-01-28T00:00:00.000,1899-12-31T17:01:00.000,...,0.98,1.00,1.00,260,260,5,2293,65211,2,146021
1,202101,Carga Normal,9604,MANUFACTURAS DIVERSAS-TAMICES/CEDAZOS Y CRIBAS...,350,Kilogramos,2021-01-26T00:00:00.000,1899-12-31T19:59:00.000,2021-01-28T00:00:00.000,1899-12-31T17:59:00.000,...,0.98,1.00,0.00,260,260,5,2293,65211,2,146021
2,202101,Carga Normal,9604,MANUFACTURAS DIVERSAS-TAMICES/CEDAZOS Y CRIBAS...,40,Kilogramos,2021-01-26T00:00:00.000,1899-12-31T19:10:00.000,2021-01-28T00:00:00.000,1899-12-31T17:10:00.000,...,0.98,0.98,0.98,260,260,5,2293,65211,2,146021
3,202101,Carga Normal,9604,MANUFACTURAS DIVERSAS-TAMICES/CEDAZOS Y CRIBAS...,40,Kilogramos,2021-01-26T00:00:00.000,1899-12-31T19:11:00.000,2021-01-28T00:00:00.000,1899-12-31T17:11:00.000,...,0.98,1.00,0.98,260,260,5,2293,65211,2,146021
4,202101,Carga Normal,9604,MANUFACTURAS DIVERSAS-TAMICES/CEDAZOS Y CRIBAS...,40,Kilogramos,2021-01-26T00:00:00.000,1899-12-31T19:02:00.000,2021-01-28T00:00:00.000,1899-12-31T17:03:00.000,...,1.00,0.98,0.98,260,260,5,2293,65211,2,146021


# 3. Profiling

Before writing any analysis query I want to know three things: whether the numeric fields convert cleanly, what the vehicle configurations look like, and how often the agreed and paid values are filled in. `TRY_CAST` returns NULL instead of failing when a value doesn't convert, so the count of NULLs after casting shows how dirty each field is.

In [3]:
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE VIEW trips_raw AS SELECT * FROM read_parquet('{RAW_PATH.as_posix()}')")

numeric_fields = [
    "horas_espera_cargue", "horas_cargue", "horas_espera_descargue",
    "horas_descargue", "horas_viaje", "valor_pactado", "valor_pagado",
]
profile_sql = "SELECT COUNT(*) AS total_rows, " + ", ".join(
    f"COUNT(*) FILTER (WHERE TRY_CAST(REPLACE({c}, ',', '.') AS DOUBLE) IS NULL) AS null_{c}"
    for c in numeric_fields
) + " FROM trips_raw"
print(con.execute(profile_sql).df().T)

print("\n--- Months in the file ---")
print(con.execute("SELECT a_o_mes, COUNT(*) AS records FROM trips_raw GROUP BY a_o_mes").df())

print("\n--- Hours: basic distribution ---")
print(con.execute("""
    SELECT
        quantile_cont(TRY_CAST(REPLACE(horas_espera_cargue, ',', '.') AS DOUBLE), [0.5, 0.9, 0.99]) AS espera_cargue_p50_p90_p99,
        quantile_cont(TRY_CAST(REPLACE(horas_espera_descargue, ',', '.') AS DOUBLE), [0.5, 0.9, 0.99]) AS espera_descargue_p50_p90_p99,
        quantile_cont(TRY_CAST(REPLACE(horas_viaje, ',', '.') AS DOUBLE), [0.5, 0.9, 0.99]) AS viaje_p50_p90_p99
    FROM trips_raw
""").df().T)

print("\n--- Vehicle configurations ---")
print(con.execute("""
    SELECT configuracion, COUNT(*) AS records,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM trips_raw
    GROUP BY configuracion
    ORDER BY records DESC
    LIMIT 15
""").df())

print("\n--- Sample of raw freight values ---")
print(con.execute("SELECT valor_pactado, valor_pagado FROM trips_raw LIMIT 10").df())

                                  0
total_rows                   836013
null_horas_espera_cargue          0
null_horas_cargue                 0
null_horas_espera_descargue       0
null_horas_descargue              0
null_horas_viaje                  0
null_valor_pactado                0
null_valor_pagado                 0

--- Months in the file ---
  a_o_mes  records
0  202101   836013

--- Hours: basic distribution ---
                                                0
espera_cargue_p50_p90_p99        [0.0, 1.0, 4.85]
espera_descargue_p50_p90_p99      [0.0, 1.0, 7.5]
viaje_p50_p90_p99             [25.0, 82.5, 191.0]

--- Vehicle configurations ---
   configuracion  records    pct
0              2   387403  46.34
1            3S3   235675  28.19
2             CA    84878  10.15
3            3S2    47630   5.70
4            2S2    37000   4.43
5              3    28781   3.44
6            2S3     9002   1.08
7             V3     1243   0.15
8            2R2     1074   0.13
9            

All 836,013 rows downloaded, all of them from a single month (`a_o_mes` = 202101), and every hour and value field converts to a number with no nulls. The hours look unusual from the start: the median waiting time is 0 at both loading and unloading, and even the 90th percentile is one hour. Trip duration is more spread out (median 25 hours, 99th percentile 191).

Two codes dominate the records: `2` (two-axle rigid truck, 46%) and `3S3` (three-axle tractor with a three-axle semi-trailer, 28%). A third code, `CA`, covers 10% of records and doesn't follow the axle pattern. These are shares of rows; Section 4.1 shows why rows and trips aren't the same thing here.

The freight values carry no thousands separators, so they convert as they are. Their scale is odd, though: the same sample mixes values like 260 and 920 with 1,400,000, which suggests some companies report in thousands of pesos and others in pesos. That rules out comparing amounts across trips. Agreed vs paid within the same trip is still valid, since both come from the same record.

# 4. Typed view and plausibility

Every field arrives as text, so the first step is a typed view over the raw file. Before trusting the hours, I looked at how many values are negative or implausibly long. A trip record with a 30-day loading wait is more likely a typing error in the manifest than a real truck waiting a month.

In [4]:
hour_fields = ["horas_espera_cargue", "horas_cargue", "horas_espera_descargue", "horas_descargue", "horas_viaje"]

plausibility = []
for field in hour_fields:
    row = con.execute(f"""
        SELECT
            '{field}' AS field,
            COUNT(*) FILTER (WHERE h < 0) AS negative,
            COUNT(*) FILTER (WHERE h > 72) AS over_3_days,
            COUNT(*) FILTER (WHERE h > 240) AS over_10_days,
            MAX(h) AS max_hours
        FROM (SELECT TRY_CAST(REPLACE({field}, ',', '.') AS DOUBLE) AS h FROM trips_raw)
    """).df()
    plausibility.append(row)

pd.concat(plausibility, ignore_index=True)

,field,negative,over_3_days,over_10_days,max_hours
0,horas_espera_cargue,0,40,1,240.98
1,horas_cargue,776,239,1,337.00
2,horas_espera_descargue,0,231,5,336.98
3,horas_descargue,0,557,17,312.00
4,horas_viaje,0,104573,4864,727.00


The two waiting fields are almost clean: no negatives, and only 1 and 5 values above 240 hours. The problems are elsewhere. `horas_cargue` has 776 negative values (I don't use that field), and trip duration has a long tail, with 104,573 records (12.5%) above three days and 4,864 above ten. Ten days is more than any trip within Colombia should take, so those are probably date-entry errors.

I keep records whose waiting and trip hours fall between 0 and 240. The rule only removes values that can't be real, about 0.6% of records (0.4% of trips, Section 4.2), and the medians and percentiles below are robust to whatever extreme values remain.

Vehicle configurations follow the Colombian axle code: a number for the axles of the truck, an `S` when it pulls a semi-trailer (a tractor-trailer, what everyone here calls a *tractomula*), and an `R` or `B` when it pulls a full or balanced trailer. Codes made only of digits are rigid trucks, and I count `V2` and `V3` with them, since they appear to be dump trucks (*volquetas*). `CA` doesn't follow the axle pattern and I couldn't confirm what it stands for, so it stays as its own class.

## 4.1 Consignments, not trips

The first five rows of the file share the same plate, driver, company, dates and agreed value (260), and all five say `cantidad_remesas_viaje = 5`. That made me suspect a row is a consignment (*remesa*), one of several loads carried on the same trip. The Ministry's own rules point the same way: the annex to Resolución 0377 de 2013 records logistic times per *remesa* and totals them in the trip manifest.

The file has no trip ID, so I tested the idea with what it does have: the reported number of consignments per trip, and how many rows share the same company, plate, driver and loading and unloading dates.

In [5]:
print("--- Rows by reported number of consignments in the trip ---")
print(con.execute("""
    SELECT
        TRY_CAST(cantidad_remesas_viaje AS INTEGER) AS consignments_in_trip,
        COUNT(*) AS rows,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_rows
    FROM trips_raw
    GROUP BY 1
    ORDER BY rows DESC
    LIMIT 8
""").df().to_string(index=False))

trip_key = "empresa_transporte, placa, conductor, fechasalidacargue, fechallegadadescargue"

print("\n--- Rows sharing the same company, plate, driver and dates ---")
print(con.execute(f"""
    WITH key_groups AS (
        SELECT
            {trip_key},
            COUNT(*) AS n_rows,
            MIN(TRY_CAST(cantidad_remesas_viaje AS INTEGER)) AS k_min,
            MAX(TRY_CAST(cantidad_remesas_viaje AS INTEGER)) AS k_max,
            COUNT(DISTINCT valor_pactado) AS agreed_values,
            COUNT(DISTINCT codigo_descargue) AS destinations
        FROM trips_raw
        WHERE TRY_CAST(cantidad_remesas_viaje AS INTEGER) > 0
        GROUP BY ALL
    )
    SELECT
        COUNT(*) AS key_groups,
        ROUND(100.0 * AVG((n_rows = k_min AND k_min = k_max)::INT), 1) AS pct_rows_match_reported_count,
        ROUND(100.0 * AVG((n_rows > k_max AND n_rows % k_min = 0)::INT), 1) AS pct_several_trips_same_key,
        ROUND(100.0 * AVG((n_rows < k_min)::INT), 1) AS pct_fewer_rows_than_reported,
        ROUND(100.0 * AVG((agreed_values > 1)::INT) FILTER (WHERE n_rows = k_min AND k_min > 1), 1) AS pct_multi_trips_with_2plus_agreed_values,
        ROUND(100.0 * AVG((destinations > 1)::INT) FILTER (WHERE n_rows = k_min AND k_min > 1), 1) AS pct_multi_trips_with_2plus_destinations
    FROM key_groups
""").df().T)

--- Rows by reported number of consignments in the trip ---
 consignments_in_trip   rows  pct_rows
                    1 621127      74.3
                    2  51524       6.2
                   20  29362       3.5
                    3  22993       2.8
                    4  15653       1.9
                    5  13027       1.6
                    6   9858       1.2
                    7   9267       1.1

--- Rows sharing the same company, plate, driver and dates ---
                                                 0
key_groups                                632195.0
pct_rows_match_reported_count                 91.6
pct_several_trips_same_key                     5.8
pct_fewer_rows_than_reported                   2.4
pct_multi_trips_with_2plus_agreed_values       0.2
pct_multi_trips_with_2plus_destinations       34.5


Three quarters of the rows (74.3%) come from single-consignment trips, but the remaining quarter is spread across trips of up to 20 or more consignments, and a trip with 20 shows up 20 times.

The key test supports reading rows as consignments. In 91.6% of the groups that share company, plate, driver and dates, the number of rows equals the reported number of consignments. Another 5.8% hold an exact multiple of it, which looks like the same truck and driver making two or more trips with the same dates. In 2.4% there are fewer rows than reported, so some consignments are either missing or fall outside January.

Inside multi-consignment trips, the agreed value is the same on every row in 99.8% of cases, so freight value belongs to the trip. The destination, on the other hand, changes in about a third of them, and the waiting and trip hours also vary between rows: they're recorded per stop.

That rules out collapsing rows into trips with this key. It would merge different trips in about one group in twelve, and taking a maximum or a first value of each field would mix stops. I keep one row per consignment and weight it by 1/k, where k is `cantidad_remesas_viaje`: a trip reported with five consignments contributes five rows of 0.2 each, and every trip adds up to one. The ten rows that report zero consignments can't be weighted, so they're excluded.

## 4.2 Typed view with trip weights

The typed view keeps one row per consignment and adds `trip_weight` = 1 / `cantidad_remesas_viaje`. Counts become `SUM(trip_weight)`, shares are weighted the same way, and the vehicle classes from Section 4 are built here too.

Percentiles need the weight as well, and DuckDB's `quantile_cont` doesn't take one. The two helpers below build a weighted percentile from the cumulative share of trips: the first value at which that share reaches the quantile. It doesn't interpolate between values, which makes no visible difference at one decimal.

In [6]:
MAX_HOURS = 240

con.execute(f"""
    CREATE OR REPLACE VIEW trips AS
    WITH typed AS (
        SELECT
            cargue,
            descargue,
            producto,
            configuracion,
            naturaleza,
            empresa_transporte,
            TRY_CAST(cantidad_remesas_viaje AS INTEGER) AS consignments_in_trip,
            TRY_CAST(REPLACE(horas_espera_cargue, ',', '.') AS DOUBLE) AS wait_loading_h,
            TRY_CAST(REPLACE(horas_cargue, ',', '.') AS DOUBLE) AS loading_h,
            TRY_CAST(REPLACE(horas_espera_descargue, ',', '.') AS DOUBLE) AS wait_unloading_h,
            TRY_CAST(REPLACE(horas_descargue, ',', '.') AS DOUBLE) AS unloading_h,
            TRY_CAST(REPLACE(horas_viaje, ',', '.') AS DOUBLE) AS trip_h,
            TRY_CAST(REPLACE(valor_pactado, ',', '.') AS DOUBLE) AS agreed_value,
            TRY_CAST(REPLACE(valor_pagado, ',', '.') AS DOUBLE) AS paid_value
        FROM trips_raw
    )
    SELECT
        *,
        1.0 / consignments_in_trip AS trip_weight,
        wait_loading_h + wait_unloading_h AS total_wait_h,
        CASE
            WHEN configuracion IS NULL OR TRIM(configuracion) = '' THEN 'Unknown'
            WHEN UPPER(configuracion) LIKE '%S%' THEN 'Tractor-trailer (tractomula)'
            WHEN UPPER(configuracion) LIKE '%R%' OR UPPER(configuracion) LIKE '%B%' THEN 'Truck with trailer'
            WHEN regexp_full_match(configuracion, '[0-9]+') OR UPPER(configuracion) LIKE 'V%' THEN 'Rigid truck'
            ELSE 'Other code (CA)'
        END AS vehicle_class,
        (wait_loading_h BETWEEN 0 AND {MAX_HOURS}
         AND wait_unloading_h BETWEEN 0 AND {MAX_HOURS}
         AND trip_h BETWEEN 0 AND {MAX_HOURS}) AS valid_hours
    FROM typed
    WHERE consignments_in_trip > 0
""")


def cum_share(value, partition=None):
    """Cumulative share of trips up to each value, within an optional partition."""
    part = f"PARTITION BY {partition} " if partition else ""
    return (f"SUM(trip_weight) OVER ({part}ORDER BY {value} ROWS UNBOUNDED PRECEDING) "
            f"/ SUM(trip_weight) OVER ({part.strip()}) AS cs_{value}")


def weighted_quantile(value, q):
    """First value whose cumulative share of trips reaches q (needs cum_share in a previous CTE)."""
    return f"MIN({value}) FILTER (WHERE cs_{value} >= {q})"


print(con.execute("""
    SELECT COUNT(*) AS consignments, ROUND(SUM(trip_weight)) AS trips,
           ROUND(100.0 * SUM(trip_weight) FILTER (WHERE NOT valid_hours) / SUM(trip_weight), 2) AS pct_trips_removed_by_hours_rule
    FROM trips
""").df().to_string(index=False))

con.execute("""
    SELECT
        vehicle_class,
        COUNT(*) AS consignments,
        ROUND(SUM(trip_weight)) AS trips,
        ROUND(SUM(trip_weight) FILTER (WHERE valid_hours)) AS trips_valid_hours,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_consignments,
        ROUND(100.0 * SUM(trip_weight) / SUM(SUM(trip_weight)) OVER (), 1) AS pct_of_trips
    FROM trips
    GROUP BY vehicle_class
    ORDER BY trips DESC
""").df()

 consignments    trips  pct_trips_removed_by_hours_rule
       836003 670409.0                             0.38


,vehicle_class,consignments,trips,trips_valid_hours,pct_of_consignments,pct_of_trips
0,Rigid truck,418967,310212.0,308749.0,50.1,46.3
1,Tractor-trailer (tractomula),330254,302597.0,301679.0,39.5,45.1
2,Other code (CA),84878,56128.0,55946.0,10.2,8.4
3,Truck with trailer,1904,1472.0,1468.0,0.2,0.2


The 836,003 weighted records add up to about 670,400 trips. The fleet shares move with the weights: tractomulas go from 39.5% of records to 45.1% of trips, and rigid trucks from 50.1% to 46.3%. Rigid trucks and the `CA` class run more multi-drop trips, so counting rows overstates them.

The hours rule removes 0.38% of trips. There's no `Unknown` class this month (every record has a configuration code), so the filter on it below changes nothing. I keep it in case another month has blank codes.

# 5. Waiting time by vehicle class

Waiting is the part of a trip that adds cost without moving cargo: the truck is at the loading or unloading site, but nothing is happening yet. I compare waiting at origin and destination by vehicle class, using medians and the 90th percentile, since a few trips with very long waits would pull an average up. Everything is weighted by trip.

Waiting is recorded per consignment, so a trip with three drops has three unloading waits. With the weights, each of those counts as a third of a trip, and the figures read as the typical wait per stop, with every trip weighing the same.

Before comparing classes, a check on the waiting fields themselves. With a median of zero and a 90th percentile of one hour, I suspected the values are often filled with a default instead of being measured. If that's the case, a handful of exact values should cover most trips.

In [7]:
for field in ["horas_espera_cargue", "horas_espera_descargue"]:
    print(f"--- Most frequent values: {field} ---")
    print(con.execute(f"""
        SELECT
            {field} AS value,
            COUNT(*) AS records,
            ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_records,
            ROUND(100.0 * SUM(1.0 / TRY_CAST(cantidad_remesas_viaje AS INTEGER))
                  / SUM(SUM(1.0 / TRY_CAST(cantidad_remesas_viaje AS INTEGER))) OVER (), 1) AS pct_trips
        FROM trips_raw
        WHERE TRY_CAST(cantidad_remesas_viaje AS INTEGER) > 0
        GROUP BY {field}
        ORDER BY records DESC
        LIMIT 10
    """).df().to_string(index=False))
    print()

--- Most frequent values: horas_espera_cargue ---
value  records  pct_records  pct_trips
 0.00   441299         52.8       49.6
 0.98    73643          8.8        9.9
 1.00    45889          5.5        6.2
 0.33    33189          4.0        3.9
 0.23    23061          2.8        2.6
 0.50    21773          2.6        3.0
 0.17    18651          2.2        2.4
 0.08    16884          2.0        2.3
 0.48    14789          1.8        2.1
 0.02    14118          1.7        1.7

--- Most frequent values: horas_espera_descargue ---
value  records  pct_records  pct_trips
 0.00   418596         50.1       48.9
 0.98    76978          9.2       10.3
 1.00    44765          5.4        5.9
 0.50    34731          4.2        3.1
 0.17    24217          2.9        2.7
 0.25    22091          2.6        2.7
 0.23    21535          2.6        2.3
 0.02    20034          2.4        2.5
 2.00    14519          1.7        1.8
 0.48    12457          1.5        1.7



About half of all trips record exactly zero waiting, at loading (49.6% of trips, 52.8% of records) and at unloading (48.9% and 50.1%). The next most common values are 0.98 and 1.00 hours, around 16% of trips on each side. Real waits measured to the minute wouldn't pile up on 59 minutes and one hour like that; it looks like a default the manifest software fills in, or a value entered by habit. Together, those three values cover about two thirds of the records, and also about two thirds of the trips once they're weighted.

So these fields measure what gets reported, which is probably less than what happens. I keep the comparison below, but I read it as a comparison of recorded waits.

In [8]:
con.execute(f"""
    WITH base AS (
        SELECT * FROM trips WHERE valid_hours AND vehicle_class <> 'Unknown'
    ),
    with_shares AS (
        SELECT
            *,
            {cum_share('wait_loading_h', 'vehicle_class')},
            {cum_share('wait_unloading_h', 'vehicle_class')},
            {cum_share('total_wait_h', 'vehicle_class')},
            {cum_share('trip_h', 'vehicle_class')}
        FROM base
    )
    SELECT
        vehicle_class,
        ROUND(SUM(trip_weight)) AS trips,
        ROUND({weighted_quantile('wait_loading_h', 0.5)}, 1) AS median_wait_loading_h,
        ROUND({weighted_quantile('wait_unloading_h', 0.5)}, 1) AS median_wait_unloading_h,
        ROUND({weighted_quantile('total_wait_h', 0.5)}, 1) AS median_total_wait_h,
        ROUND({weighted_quantile('total_wait_h', 0.9)}, 1) AS p90_total_wait_h,
        ROUND({weighted_quantile('trip_h', 0.5)}, 1) AS median_trip_h,
        ROUND(100.0 * SUM(trip_weight * (total_wait_h > trip_h)::INT) / SUM(trip_weight), 1) AS pct_waiting_longer_than_driving
    FROM with_shares
    GROUP BY vehicle_class
    ORDER BY trips DESC
""").df()

,vehicle_class,trips,median_wait_loading_h,median_wait_unloading_h,median_total_wait_h,p90_total_wait_h,median_trip_h,pct_waiting_longer_than_driving
0,Rigid truck,308749.0,0.0,0.0,0.2,2.0,23.0,1.3
1,Tractor-trailer (tractomula),301679.0,0.0,0.0,0.1,3.1,30.3,1.4
2,Other code (CA),55946.0,0.2,0.2,0.6,2.0,18.0,2.6
3,Truck with trailer,1468.0,0.0,0.0,0.1,3.0,22.0,0.2


On recorded waits, vehicle classes barely differ. The median total wait is between 0.1 and 0.6 hours for every class, and only 1 to 3% of trips record more waiting than driving. The highest median belongs to the `CA` class (0.6 hours, about a quarter hour at each end), on its shorter trips (median 18 hours).

The visible difference is in the tail: tractomulas have a 90th percentile of 3.1 hours against 2.0 for rigid trucks, which fits with the bulk-cargo corridors in Section 6, where the long waits concentrate. Tractomulas also drive longer trips (median 30 hours against 23), as expected for long-haul vehicles. Truck-with-trailer is too small (about 1,500 trips) to read much into.

Given the check above, I wouldn't conclude that Colombian trucks barely wait. What the data supports is narrower: recorded waits are short and similar across vehicle types.

# 6. Corridors with the longest waits

A corridor is a pair of loading and unloading municipalities. I keep corridors with at least 200 trips in the month and look separately at corridors that start or end in Medellín. `strip_accents` handles "MEDELLÍN" and "MEDELLIN" written both ways.

Origin and destination belong to the consignment, and a third of multi-consignment trips deliver to more than one place. A trip with three drops therefore counts as a third of a trip in each of its three corridors.

I rank corridors by the 90th percentile of total waiting time and keep the median beside it. With half the trips recording zero waiting, the median ties hundreds of corridors at zero, and the long waits that matter for an operator live in the tail. The most common product per corridor goes in the table too, as a check on what each corridor carries.

In [9]:
corridor_cte = f"""
WITH base AS (
    SELECT * FROM trips WHERE valid_hours
),
with_shares AS (
    SELECT
        *,
        {cum_share('total_wait_h', 'cargue, descargue')},
        {cum_share('trip_h', 'cargue, descargue')}
    FROM base
),
corridor_stats AS (
    SELECT
        cargue,
        descargue,
        SUM(trip_weight) AS trips,
        {weighted_quantile('total_wait_h', 0.5)} AS median_total_wait_h,
        {weighted_quantile('total_wait_h', 0.9)} AS p90_total_wait_h,
        {weighted_quantile('trip_h', 0.5)} AS median_trip_h,
        LEFT(mode(producto), 45) AS main_product
    FROM with_shares
    GROUP BY cargue, descargue
    HAVING SUM(trip_weight) >= 200
),
ranked AS (
    SELECT
        *,
        RANK() OVER (ORDER BY p90_total_wait_h DESC) AS wait_rank,
        COUNT(*) OVER () AS n_corridors
    FROM corridor_stats
)
"""

print("--- Top 15 corridors by 90th percentile of waiting time ---")
print(con.execute(corridor_cte + """
    SELECT wait_rank, cargue, descargue, ROUND(trips) AS trips,
           ROUND(p90_total_wait_h, 1) AS p90_wait_h,
           ROUND(median_total_wait_h, 1) AS median_wait_h,
           ROUND(median_trip_h, 1) AS median_trip_h,
           main_product
    FROM ranked
    ORDER BY wait_rank, trips DESC
    LIMIT 15
""").df().to_string(index=False))

print("\n--- Corridors touching Medellín (15 busiest) ---")
print(con.execute(corridor_cte + """
    SELECT wait_rank, n_corridors, cargue, descargue, ROUND(trips) AS trips,
           ROUND(p90_total_wait_h, 1) AS p90_wait_h,
           ROUND(median_total_wait_h, 1) AS median_wait_h,
           ROUND(median_trip_h, 1) AS median_trip_h
    FROM ranked
    WHERE strip_accents(UPPER(cargue)) LIKE '%MEDELLIN%'
       OR strip_accents(UPPER(descargue)) LIKE '%MEDELLIN%'
    ORDER BY trips DESC
    LIMIT 15
""").df().to_string(index=False))

--- Top 15 corridors by 90th percentile of waiting time ---
 wait_rank                              cargue                    descargue  trips  p90_wait_h  median_wait_h  median_trip_h                                  main_product
         1           PUERTO LIBERTADOR CORDOBA       SANTIAGO DE TOLU SUCRE  738.0        48.0           26.0          103.0 COMBUSTIBLES MINERALES-ACEITES MINERALES Y PR
         2                        SOCHA BOYACA BUENAVENTURA VALLE DEL CAUCA  256.0        46.6            2.3           66.0 COMBUSTIBLES MINERALES-ACEITES MINERALES Y PR
         3                        SOCHA BOYACA    CUCUTA NORTE DE SANTANDER  214.0        44.1            0.0           66.0 COMBUSTIBLES MINERALES-ACEITES MINERALES Y PR
         4              BARRANQUILLA ATLANTICO           FUNZA CUNDINAMARCA  477.0        39.1            0.0           44.0 PRODUCTOS CERÁMICOS-VAJILLAS Y DEMAS ARTICULO
         5                    GALAPA ATLANTICO            CARTAGENA BOLIVAR  698.0   

Ranked by the 90th percentile, the top of the list is bulk freight. Six of the top 15 corridors carry mineral fuels (coal and coke, the truncated `COMBUSTIBLES MINERALES` label) from mines in Córdoba, Boyacá, Cundinamarca and Norte de Santander to ports or border cities, two carry crude oil out of Arauca and Meta, and three carry maize. Puerto Libertador (Córdoba) to Santiago de Tolú (Sucre), from a mining area to the ports of the Gulf of Morrosquillo, still leads by far: a median wait of 26 hours, a 90th percentile of 48 and a median trip of 103 hours over 738 trips.

A ranking by median would hide most of these corridors: Socha to Cúcuta, El Zulia to Cartagena and Barranquilla to Funza all have a median wait of zero, but one trip in ten waits more than 20 hours. 217 of the 579 corridors have a median of exactly zero, so the median can't separate them.

Medellín's busiest corridors don't wait long. None of the fifteen busiest goes above a 90th percentile of 5 hours. The worst is Itagüí to Medellín (5.0 hours, rank 52 of 579), and Bogotá to Medellín, the corridor with the most trips (4,235), sits at 3.0 hours. Envigado to Medellín has more rows than any other Medellín corridor (7,024), but weighted it's 2,680 trips, because it carries many multi-drop urban deliveries.

Counting rows instead of trips, 696 corridors would pass the 200 threshold; weighted, 579 do. Multi-consignment trips stop inflating the counts, and trips with stops in several municipalities are split across their corridors.

# 7. Agreed vs paid freight value

Each manifest records the value agreed with the transport company and the value finally paid. Paying less than agreed is a known complaint in Colombian trucking, and this is the one question the U.S. flight data had no equivalent for.

Both values belong to the trip: they're identical across the consignments of 99.8% of multi-consignment trips (Section 4.1). Counted by row, a trip with 20 consignments would vote 20 times, so the shares below are weighted by trip. I only compare trips where both values are positive.

In [10]:
con.execute("""
    WITH valued AS (
        SELECT
            vehicle_class,
            trip_weight,
            CASE
                WHEN paid_value < agreed_value THEN 'Paid less'
                WHEN paid_value = agreed_value THEN 'Paid as agreed'
                ELSE 'Paid more'
            END AS payment_status,
            100.0 * (agreed_value - paid_value) / agreed_value AS shortfall_pct
        FROM trips
        WHERE agreed_value > 0 AND paid_value > 0 AND vehicle_class <> 'Unknown'
    ),
    shortfalls AS (
        SELECT
            vehicle_class,
            shortfall_pct,
            SUM(trip_weight) OVER (PARTITION BY vehicle_class ORDER BY shortfall_pct ROWS UNBOUNDED PRECEDING)
                / SUM(trip_weight) OVER (PARTITION BY vehicle_class) AS cs_class,
            SUM(trip_weight) OVER (ORDER BY shortfall_pct ROWS UNBOUNDED PRECEDING)
                / SUM(trip_weight) OVER () AS cs_all
        FROM valued
        WHERE payment_status = 'Paid less'
    ),
    median_shortfall AS (
        SELECT vehicle_class, MIN(shortfall_pct) FILTER (WHERE cs_class >= 0.5) AS median_shortfall
        FROM shortfalls GROUP BY vehicle_class
        UNION ALL
        SELECT 'All classes', MIN(shortfall_pct) FILTER (WHERE cs_all >= 0.5) FROM shortfalls
    ),
    shares AS (
        SELECT
            COALESCE(vehicle_class, 'All classes') AS vehicle_class,
            SUM(trip_weight) AS trips,
            100.0 * SUM(trip_weight) FILTER (WHERE payment_status = 'Paid less') / SUM(trip_weight) AS pct_paid_less,
            100.0 * SUM(trip_weight) FILTER (WHERE payment_status = 'Paid as agreed') / SUM(trip_weight) AS pct_paid_as_agreed,
            100.0 * SUM(trip_weight) FILTER (WHERE payment_status = 'Paid more') / SUM(trip_weight) AS pct_paid_more
        FROM valued
        GROUP BY ROLLUP (vehicle_class)
    )
    SELECT
        s.vehicle_class,
        ROUND(s.trips) AS trips,
        ROUND(s.pct_paid_less, 2) AS pct_paid_less,
        ROUND(s.pct_paid_as_agreed, 2) AS pct_paid_as_agreed,
        ROUND(s.pct_paid_more, 2) AS pct_paid_more,
        ROUND(m.median_shortfall, 1) AS median_shortfall_pct_when_less
    FROM shares AS s
    LEFT JOIN median_shortfall AS m USING (vehicle_class)
    ORDER BY trips DESC
""").df()

,vehicle_class,trips,pct_paid_less,pct_paid_as_agreed,pct_paid_more,median_shortfall_pct_when_less
0,All classes,538851.0,0.86,97.76,1.38,4.2
1,Rigid truck,266414.0,0.39,98.83,0.78,4.4
2,Tractor-trailer (tractomula),219614.0,1.43,96.25,2.32,5.4
3,Other code (CA),52132.0,0.83,98.72,0.44,0.0
4,Truck with trailer,690.0,6.08,93.77,0.14,2.0


Agreed and paid values match on 97.76% of trips (the first row, "All classes", is the total that `ROLLUP` adds). Underpayment is rare (0.86%) and so is overpayment (1.38%). Tractomulas are paid less than agreed more often than rigid trucks (1.43% vs 0.39%), and by more when it happens (median shortfall of 5.4% against 4.4%). They're also the large class most often paid *more* (2.32%), so what stands out is that their payments move from the agreed value more often, in both directions. Truck-with-trailer has the highest share paid less (6.08%), but on 690 trips.

The `CA` class has a median shortfall of zero for a different reason. When I looked at its underpaid records, the typical gap was a thousandth of a unit (249.981 agreed, 249.98 paid): rounding in the reported amounts. Across all classes, about one in five underpaid trips differs by less than 0.1%. I kept the classification exact so these shares can be checked against the raw data; a tolerance would be the obvious next refinement.

This doesn't match the sector's reputation, where paying less than agreed is a common complaint. The likely reason is the source: the companies register these values themselves in the RNDC, and a payment dispute would rarely show up in a field filled in by the paying side.

# 8. Conclusions

The most useful findings here are about the data. First, each row is a consignment, and trips can span many rows: a quarter of the rows belong to trips with several consignments, so counting rows overstates multi-drop vehicles and corridors. Weighting each record by 1/k fixes the counts without inventing a trip ID the file doesn't have. Second, two thirds of the waiting records hold one of three values (0, 0.98 or 1 hour), so the RNDC captures waiting poorly, and any analysis of logistics times built on these fields would understate real waits. Checking the unit of each row and the most frequent values of a time field are both things I'd repeat with any manifest-based source.

Within those limits, long recorded waits concentrate on a few bulk-cargo corridors, mostly coal, coke and crude oil, led by far by a mining-to-port route between Córdoba and Sucre, while Medellín's busiest corridors stay at or below 5 hours even in their tail. Tractomulas show longer tails of waiting than rigid trucks. On payments, they're paid less than agreed more often than rigid trucks, and paid above it more often too, still on fewer than 3% of their trips either way.

All of this is one month, January 2021. Nothing here says whether these patterns are seasonal, persistent or improving, which is why the main analysis of this project uses the U.S. flight data.